# Non-stationary CartPole (oscillating pole length)

This example uses [NS-Gym](https://github.com/scope-lab-vu/ns_gym) — an external framework for non-stationary MDPs (we use it; we did not build it). mouse-env has **no NS-Gym integration code**: you build the NS-Gym env yourself in an `env_fn` factory and apply a small adapter that flattens NS-Gym's dict observation and exposes `info["ns_params"]`. mouse-env then surfaces those values in `result[i]["ns_params"]`, and you still call the usual `step()` API.

Requires the optional `non-stationary` extra: `pip install 'mouse-env[non-stationary]'` (`ns_gym`).

In [ ]:
%load_ext autoreload
%autoreload 2

from typing import Any

import gymnasium as gym
import numpy as np
from ns_gym.schedulers import ContinuousScheduler
from ns_gym.update_functions import OscillatingUpdate
from ns_gym.wrappers import NSClassicControlWrapper

from mouse_envs import EnvConfig, make_vector_env

## Adapter + factory

`NSAdapter` flattens NS-Gym's `{"state": ...}` observation and turns its ground-truth change info into `info["ns_params"]`. The factory builds a `ContinuousScheduler` + `OscillatingUpdate` for the pole `length` and wraps CartPole with `NSClassicControlWrapper`. See the [NS-Gym docs](https://nsgym.io/) for other schedulers and update functions.

In [ ]:
class NSAdapter(gym.Wrapper):
    """Flatten NS-Gym dict obs to its ``state`` and expose changes as ns_params."""

    def __init__(self, env):
        super().__init__(env)
        space = env.observation_space
        if isinstance(space, gym.spaces.Dict) and "state" in space.spaces:
            self.observation_space = space["state"]

    @staticmethod
    def _adapt(obs, info):
        state = obs["state"] if isinstance(obs, dict) and "state" in obs else obs
        env_change = info.get("Ground Truth Env Change", {}) or {}
        delta = info.get("Ground Truth Delta Change", {}) or {}
        ns_params: dict[str, Any] = {}
        for key, flag in env_change.items():
            if not key.startswith("_"):
                ns_params[key] = np.asarray(delta.get(key, 0))
                ns_params[f"{key}_flag"] = np.asarray(flag)
        return state, {"ns_params": ns_params}

    def reset(self, *, seed=None, options=None):
        obs, info = self.env.reset(seed=seed, options=options)
        return self._adapt(obs, info)

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        state, info = self._adapt(obs, info)
        return state, reward, terminated, truncated, info


def make_ns_cartpole():
    update_fns = {"length": OscillatingUpdate(ContinuousScheduler(), delta=0.01)}
    base = gym.make("CartPole-v1", max_episode_steps=500)
    ns_env = NSClassicControlWrapper(
        base, update_fns, change_notification=True, delta_change_notification=True
    )
    return NSAdapter(ns_env)

## Build and rollout

In [ ]:
cfg = EnvConfig(
    id="CartPole-ns",
    seed=42,
    num_envs=2,
    max_episode_steps=500,
    env_fn=make_ns_cartpole,
)
env = make_vector_env(cfg)

for step in range(500):
    result, metrics = env.step(env.sample_random_actions())

    if step % 50 == 0:
        print(f"step={step:3d}  ns_params={result[0].get('ns_params')}")

In [ ]:
env.close()